# UKB case examples with SuSiE-inf (data processing)

The two UKB loci of Fig 5E (chr12, HLR %) and Fig S11C (chr22, monocyte %), where the ADSP sketch LD gives SuSiE-RSS several wrong credible sets, refitted with SuSiE-inf (`unmappable_effects = "inf"`) with and without the R-uncertainty adjustment of SuSiE-rRSS (`R_finite = B_sketch`, `R_mismatch = "eb_mix"`). A check that SuSiE-inf runs with the adjustment, for the Discussion; no simulation.

All fits use the GitHub version of susieR (`stephenslab/susieR`), which has both. Same inputs as `Fig5E_5F_S11C.UKB_Case_Examples` (`rss_qc/cases/eb_wrong/`). Fits per case:
- **In-sample SuSiE** and **in-sample SuSiE-inf**: UKB z and in-sample LD (reference);
- **SuSiE-RSS** and **SuSiE-rRSS**: sketch LD, as in Fig 5E;
- **SuSiE-inf** and **SuSiE-inf + rRSS**: sketch LD, without and with the adjustment.

For the flagged fits the SER fallback is applied as in the pipeline (`pip_final`, `cs_final`); `pip`, `cs` are the fit before it.

SuSiE-inf needs the p × p reference R. Upstream `add_eigen_decomposition()` treats a low-rank X (B < p) as individual-level data and fails, so R is passed when B < p. When B ≥ p, `susie_rss` forms R from X itself, so the fit is the same.

**Outputs:**
- `figure_data/UKB_SuSiE_inf_Cases_Variants.rds` (+ csv): one row per case × variant: z (sketch and in-sample), −log10 p, lead, LD with the lead (sketch and in-sample)
- `figure_data/UKB_SuSiE_inf_Cases_PIP.rds` (+ csv): one row per case × method × variant: PIP and credible set, before and after the SER fallback
- `figure_data/UKB_SuSiE_inf_Cases_Summary.rds` (+ csv): one row per case × method: credible sets, τ² (SuSiE-inf), flag, Q_art, D_CS,max, time

In [1]:
library(tidyverse)
library(MASS)
library(Matrix)
library(crayon)
library(Rfast)
library(matrixStats)
library(mixsqp)
library(reshape)

susie_up_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/susieR"
system(sprintf("git -C %s log -1 --format='susieR %%h %%cd %%s'", susie_up_dir))
source_build <- function(dir) {
  env <- new.env(parent = globalenv())
  for (f in list.files(dir, pattern = "\\.R$", full.names = TRUE)) sys.source(f, envir = env)
  env
}
up <- source_build(file.path(susie_up_dir, "R"))

## repo helpers last, so nothing from the sourced build shadows them
source("../../R/utils.R")

case_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/cases/eb_wrong/"
cases <- c(main   = "chr12:95606906-96464566.HLR_perc.rds",         # Fig 5E
           supp_1 = "chr22:17336757-18536672.Monocyte_perc.rds")    # Fig S11C

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘MASS’


The following object is masked _by_ ‘.GlobalEnv’:

    select


The following object is masked from ‘package:dplyr’:

    select


Warning message:
“package ‘Matrix’ was built under R version 4.5.3”

Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack



Attaching package: ‘crayon’


The following object is masked from ‘package:ggplot2’:

    %+%


Loading required package: Rcpp

Loadi

In [2]:
## ---- fit every model on one case file; keep PIPs, credible sets and diagnostics ----
method_labels <- c(insample       = "In-sample SuSiE",
                   insample_inf   = "In-sample SuSiE-inf",
                   susie_rss      = "SuSiE-RSS",
                   susie_rrss     = "SuSiE-rRSS",
                   susie_inf      = "SuSiE-inf",
                   susie_inf_rrss = "SuSiE-inf + rRSS")

cs_index <- function(fit, p) {
  cs <- rep(0L, p)
  for (k in seq_along(fit$sets$cs)) cs[fit$sets$cs[[k]]] <- k
  cs
}

fit_inf_case <- function(case, f) {
  d <- readRDS(file.path(case_dir, f))
  p <- length(d$z_sketch)
  ## reference LD from the sketch. susie-inf needs the p x p R: upstream add_eigen_decomposition()
  ## takes a low-rank X (B < p) for individual-level data and fails, so pass R then (as susie_rss
  ## does itself for a full-rank X; same fit for the non-inf models)
  ref <- if (nrow(d$sketch) >= ncol(d$sketch)) list(X = d$sketch) else list(R = up$safe_cor(d$sketch))
  adj <- list(R_finite = d$B_sketch, R_mismatch = "eb_mix")
  inf <- list(unmappable_effects = "inf")

  run <- function(m, args) {
    t0 <- Sys.time()
    fit <- tryCatch(suppressWarnings(do.call(up$susie_rss, c(args, list(n = d$n)))),
                    error = function(e) { cat(sprintf("[FAIL] %s %s: %s\n", case, m, conditionMessage(e))); NULL })
    if (is.null(fit)) return(NULL)
    dg   <- fit$R_finite_diagnostics
    flag <- isTRUE(dg$R_reliability_flag)
    final <- if (flag && !is.null(dg$ser_model)) dg$ser_model else fit      # pipeline result: SER model when flagged
    summ <- tibble(case = case, method = m, n_cs = length(fit$sets$cs), n_cs_final = length(final$sets$cs),
                   tau2 = fit$tau2 %||% NA_real_, sigma2 = fit$sigma2 %||% NA_real_,
                   ser_fallback = flag, Q_art = dg$Q_art %||% NA_real_,
                   D_CS_max = if (length(dg$bf_attenuation$cs_max)) max(dg$bf_attenuation$cs_max) else NA_real_,
                   time_s = as.numeric(difftime(Sys.time(), t0, units = "secs")))
    cat(sprintf("%s %s: %d CS%s, %.0f s\n", case, m, summ$n_cs,
                if (flag) sprintf(" (flagged: SER fallback, %d CS)", summ$n_cs_final) else "", summ$time_s))
    pip <- tibble(case = case, method = m, idx = seq_len(p),
                  pip = as.numeric(fit$pip), cs = cs_index(fit, p),
                  pip_final = as.numeric(final$pip), cs_final = cs_index(final, p))
    list(summ = summ, pip = pip)
  }

  res <- list(
    insample       = run("insample",       c(list(z = d$insample_z, R = d$insample_LD))),
    insample_inf   = run("insample_inf",   c(list(z = d$insample_z, R = d$insample_LD), inf)),
    susie_rss      = run("susie_rss",      c(list(z = d$z_sketch), ref)),
    susie_rrss     = run("susie_rrss",     c(list(z = d$z_sketch), ref, adj)),
    susie_inf      = run("susie_inf",      c(list(z = d$z_sketch), ref, inf)),
    susie_inf_rrss = run("susie_inf_rrss", c(list(z = d$z_sketch), ref, adj, inf))) %>% compact()

  top <- which.max(abs(d$z_sketch))
  vars <- tibble(case = case, file = f, idx = seq_len(p), z = d$z_sketch, z_insample = d$insample_z,
                 logp = -(pnorm(-abs(d$z_sketch), log.p = TRUE) + log(2)) / log(10),
                 lead = seq_len(p) == top,
                 ld_lead_sketch = d$sketch_LD[top, ], ld_lead_insample = d$insample_LD[top, ])
  list(vars = vars, summ = map_dfr(res, "summ"), pip = map_dfr(res, "pip"))
}

In [3]:
res  <- imap(cases, ~ fit_inf_case(.y, .x))
vars <- map_dfr(res, "vars")
summ <- map_dfr(res, "summ")
pip  <- map_dfr(res, "pip")

summ %>% mutate(across(where(is.double), ~ signif(.x, 3))) %>% as.data.frame()

save_data(vars, "UKB_SuSiE_inf_Cases_Variants")
save_data(pip,  "UKB_SuSiE_inf_Cases_PIP")
save_data(summ, "UKB_SuSiE_inf_Cases_Summary")

HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.




main insample: 1 CS, 9 s


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.


HINT: Switching to PIP-based convergence: unmappable effects models (inf/ash) do not have a well-defined ELBO.

HINT: unmappable_effects = 'inf' requires estimate_residual_variance = TRUE to estimate the infinitesimal variance tau^2; overriding to TRUE.



main insample_inf: 1 CS, 32 s


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.




main susie_rss: 9 CS, 34 s


... (15 outputs left out)

supp_1 susie_inf: 2 CS, 103 s


HINT: Setting max_iter = 50 for the SuSiE RSS model because slow convergence is often a sign of unstable summary-statistics fitting. To disable this message, explicitly set max_iter = 50 or another value in the susie_rss() call.

HINT: Switching to PIP-based convergence: R uncertainty inflation modifies per-variant SER likelihoods, which prevents a consistent model-level ELBO.

HINT: unmappable_effects = 'inf' requires estimate_residual_variance = TRUE to estimate the infinitesimal variance tau^2; overriding to TRUE.



supp_1 susie_inf_rrss: 1 CS, 47 s


case,method,n_cs,n_cs_final,tau2,sigma2,ser_fallback,Q_art,D_CS_max,time_s
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<lgl>,<dbl>,<dbl>,<dbl>
main,insample,1,1,NA,1.000,FALSE,NA,NA,9.32
main,insample_inf,1,1,5.76e-08,0.999,FALSE,NA,NA,31.50
main,susie_rss,9,9,NA,1.000,FALSE,NA,NA,34.10
main,susie_rrss,1,1,NA,1.000,FALSE,0.022200,0.000101,19.00
main,susie_inf,7,7,4.54e-06,0.980,FALSE,NA,NA,213.00
main,susie_inf_rrss,0,0,6.89e-06,0.981,FALSE,0.000201,NA,230.00
supp_1,insample,1,1,NA,1.000,FALSE,NA,NA,12.10
supp_1,insample_inf,1,1,1.94e-08,0.996,FALSE,NA,NA,80.40
supp_1,susie_rss,3,3,NA,1.000,FALSE,NA,NA,25.00
